<table align="left">
  <tr>
    <td><img src="fleet.png" alt="fleet of icecream trucks" width="120"/></td>
    <td align="left"><h1>Lesson 4: Monitoring and Evaluating your Agent</h1></td>
  </tr>
</table>


<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px">
<p> 💻 &nbsp; <b>Access <code>requirements.txt</code> file:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Open"</em>.

<p> ⬇ &nbsp; <b>Download Notebooks:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Download as"</em> and select <em>"Notebook (.ipynb)"</em>.</p>

<p> 📒 &nbsp; For more help, please see the <em>"Appendix – Tips, Help, and Download"</em> Lesson.</p>

</div>

<p style="background-color:#f7fff8; padding:15px; border-width:3px; border-color:#e0f0e0; border-style:solid; border-radius:6px"> 🚨
&nbsp; <b>Different Run Results:</b> The output generated by AI chat models can vary with each execution due to their dynamic, probabilistic nature. Don't be surprised if your results differ from those shown in the video.</p>

## Setup Tracing

In [1]:
PROJECT_NAME = "Customer-Success"

In [26]:
# Start Phoenix server in the background
import subprocess
import sys
import time
import os

# Adjust the port as needed
PHOENIX_PORT = 6006

# Optional: kill old Phoenix processes if they're still running
subprocess.run([sys.executable, '-m', 'phoenix.server', 'stop'], stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)

try:
    # Start Phoenix as a subprocess with the selected port
    phoenix_proc = subprocess.Popen([
        sys.executable, '-m', 'phoenix.server', 'start', f'--port={PHOENIX_PORT}'
    ], stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    print(f"Phoenix server launching on http://localhost:{PHOENIX_PORT}/ ...")
    time.sleep(2)  # Wait a bit for server to spin up
    
    # Set the environment variable for other cells to use
    os.environ['DLAI_LOCAL_URL'] = f"http://localhost:{PHOENIX_PORT}/"
    print(f"Phoenix UI available at: {os.environ['DLAI_LOCAL_URL']}")
except Exception as e:
    print("Failed to start Phoenix server:", e)
    phoenix_proc = None

# Note: The server runs as a background subprocess and will stop when the notebook kernel is restarted
# To explicitly stop Phoenix, run: phoenix_proc.terminate()

Phoenix server launching on http://localhost:6006/ ...
Phoenix UI available at: http://localhost:6006/


In [27]:
import os
from dotenv import load_dotenv, find_dotenv
from phoenix.otel import register
from openinference.instrumentation.smolagents import SmolagentsInstrumentor

tracer_provider = register(
    project_name=PROJECT_NAME,
    #endpoint= get_phoenix_endpoint() + "v1/traces"
    endpoint = os.getenv('DLAI_LOCAL_URL').format(port='6006') + "v1/traces"
)
SmolagentsInstrumentor().instrument(tracer_provider=tracer_provider)

Overriding of current TracerProvider is not allowed
Attempting to instrument while already instrumented


🔭 OpenTelemetry Tracing Details 🔭
|  Phoenix Project: Customer-Success
|  Span Processor: SimpleSpanProcessor
|  Collector Endpoint: http://localhost:6006/v1/traces
|  Transport: HTTP + protobuf
|  Transport Headers: {}
|  
|  Using a default SpanProcessor. `add_span_processor` will overwrite this default.
|  
|  ⚠️ WARNING: It is strongly advised to use a BatchSpanProcessor in production environments.
|  
|  `register` has set this TracerProvider as the global OpenTelemetry default.
|  To disable this behavior, call `register` with `set_global_tracer_provider=False`.



In [28]:
from smolagents import HfApiModel

model=HfApiModel("Qwen/Qwen2.5-Coder-32B-Instruct", provider="together")

model([{"role": "user", "content": "Hello!"}])

Exception while exporting Span.
Traceback (most recent call last):
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
        (self._dns_host, self.port),
    ...<2 lines>...
        socket_options=self.socket_options,
    )
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
    ~~~~~~~~~~~~^^^^
ConnectionRefusedError: [Errno 61] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-p

ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, content='Hello! How can I assist you today?', tool_calls=[], raw=ChatCompletionOutput(choices=[ChatCompletionOutputComplete(finish_reason='stop', index=0, message=ChatCompletionOutputMessage(role='assistant', content='Hello! How can I assist you today?', tool_call_id=None, tool_calls=[]), logprobs=None, seed=15737657616149774000)], created=1746097219, id='nsHdf7Y-4yUbBN-938ec642bf2d56e6', model='Qwen/Qwen2.5-Coder-32B-Instruct', system_fingerprint=None, usage=ChatCompletionOutputUsage(completion_tokens=10, prompt_tokens=31, total_tokens=41), object='chat.completion', prompt=[]))

In [16]:
from dotenv import load_dotenv, find_dotenv
load_dotenv() # load variables from local .env file

from huggingface_hub import login

login(os.getenv('HF_API_KEY'))

In [17]:
# This is where you can access the Phoenix UI:
print(os.environ.get('DLAI_LOCAL_URL'))

http://localhost:6006/


## Trace an agent run

In [18]:
from smolagents import CodeAgent

agent = CodeAgent(model=model, tools=[])

>Note, the following line will sometimes get a timeout on the interface to the tracing package due to the networked interface. If this happens, try it again.


In [21]:
agent.run("What is the 100th Fibonacci number?")

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is the 100th Fibonacci number?                                                                             │
│                                                                                                                 │
╰─ HfApiModel - Qwen/Qwen2.5-Coder-32B-Instruct ──────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Exception while exporting Span.
Traceback (most recent call last):
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
        (self._dns_host, self.port),
    ...<2 lines>...
        socket_options=self.socket_options,
    )
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
    ~~~~~~~~~~~~^^^^
ConnectionRefusedError: [Errno 61] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-p

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  def fibonacci(n):                                                                                                
      a, b = 0, 1                                                                                                  
      for _ in range(n):                                                                                           
          a, b = b, a + b                                                                                          
      return a                                                                                                     
                                                                                                                   
  fib_100 = fibonacci(100)                                                                                         
  final_answer(fib_100)                                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Exception while exporting Span.
Traceback (most recent call last):
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
        (self._dns_host, self.port),
    ...<2 lines>...
        socket_options=self.socket_options,
    )
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
    ~~~~~~~~~~~~^^^^
ConnectionRefusedError: [Errno 61] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-p

Out - Final answer: 354224848179261915075

Exception while exporting Span.
Traceback (most recent call last):
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
        (self._dns_host, self.port),
    ...<2 lines>...
        socket_options=self.socket_options,
    )
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
    ~~~~~~~~~~~~^^^^
ConnectionRefusedError: [Errno 61] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-p

[Step 1: Duration 2.01 seconds| Input tokens: 2,020 | Output tokens: 130]

Exception while exporting Span.
Traceback (most recent call last):
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
        (self._dns_host, self.port),
    ...<2 lines>...
        socket_options=self.socket_options,
    )
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
    ~~~~~~~~~~~~^^^^
ConnectionRefusedError: [Errno 61] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-p

354224848179261915075

In [22]:
from smolagents import HfApiModel

model=HfApiModel("Qwen/Qwen2.5-Coder-32B-Instruct", provider="together")

model([{"role": "user", "content": "Hello!"}])

Exception while exporting Span.
Traceback (most recent call last):
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
        (self._dns_host, self.port),
    ...<2 lines>...
        socket_options=self.socket_options,
    )
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
    ~~~~~~~~~~~~^^^^
ConnectionRefusedError: [Errno 61] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-p

ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, content='Hello! How can I assist you today?', tool_calls=[], raw=ChatCompletionOutput(choices=[ChatCompletionOutputComplete(finish_reason='stop', index=0, message=ChatCompletionOutputMessage(role='assistant', content='Hello! How can I assist you today?', tool_call_id=None, tool_calls=[]), logprobs=None, seed=13254543431606952000)], created=1746096330, id='nsHZ6v3-4yUbBN-938eb08fead3819f', model='Qwen/Qwen2.5-Coder-32B-Instruct', system_fingerprint=None, usage=ChatCompletionOutputUsage(completion_tokens=10, prompt_tokens=31, total_tokens=41), object='chat.completion', prompt=[]))

In [29]:
from smolagents import HfApiModel

model=HfApiModel("Qwen/Qwen2.5-Coder-32B-Instruct", provider="together")

model([{"role": "user", "content": "Hello!"}])

Exception while exporting Span.
Traceback (most recent call last):
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
        (self._dns_host, self.port),
    ...<2 lines>...
        socket_options=self.socket_options,
    )
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
    ~~~~~~~~~~~~^^^^
ConnectionRefusedError: [Errno 61] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-p

ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, content='Hello! How can I assist you today?', tool_calls=[], raw=ChatCompletionOutput(choices=[ChatCompletionOutputComplete(finish_reason='stop', index=0, message=ChatCompletionOutputMessage(role='assistant', content='Hello! How can I assist you today?', tool_call_id=None, tool_calls=[]), logprobs=None, seed=196704964311267400)], created=1746097279, id='nsHdy7d-4yUbBN-938ec7bd2956826e', model='Qwen/Qwen2.5-Coder-32B-Instruct', system_fingerprint=None, usage=ChatCompletionOutputUsage(completion_tokens=10, prompt_tokens=31, total_tokens=41), object='chat.completion', prompt=[]))

In [30]:
from smolagents import HfApiModel

model=HfApiModel("Qwen/Qwen2.5-Coder-32B-Instruct", provider="together")

model([{"role": "user", "content": "Hello!"}])

Exception while exporting Span.
Traceback (most recent call last):
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/huggingface_hub/utils/_http.py", line 409, in hf_raise_for_status
    response.raise_for_status()
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/requests/models.py", line 1024, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 402 Client Error: Payment Required for url: https://router.huggingface.co/together/v1/chat/completions

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/opentelemetry/trace/__init__.py", line 587, in use_span
    yield span
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/py

HfHubHTTPError: 402 Client Error: Payment Required for url: https://router.huggingface.co/together/v1/chat/completions (Request ID: Root=1-68135484-5aae62aa4bb3cc732a8613e4;f916d341-f2bc-4278-834c-ebfd268109c2)

You have exceeded your monthly included credits for Inference Providers. Subscribe to PRO to get 20x more monthly included credits.

In [ ]:
from smolagents import HfApiModel

model=HfApiModel("Qwen/Qwen2.5-Coder-32B-Instruct", provider="together")

model([{"role": "user", "content": "Hello!"}])

Exception while exporting Span.
Traceback (most recent call last):
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
        (self._dns_host, self.port),
    ...<2 lines>...
        socket_options=self.socket_options,
    )
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
    ~~~~~~~~~~~~^^^^
ConnectionRefusedError: [Errno 61] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-p

ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, content='Hello! How can I assist you today?', tool_calls=[], raw=ChatCompletionOutput(choices=[ChatCompletionOutputComplete(finish_reason='stop', index=0, message=ChatCompletionOutputMessage(role='assistant', content='Hello! How can I assist you today?', tool_call_id=None, tool_calls=[]), logprobs=None, seed=9779207749351244000)], created=1746095134, id='nsHSyYP-4yUbBN-938e935fba58c9b8', model='Qwen/Qwen2.5-Coder-32B-Instruct', system_fingerprint=None, usage=ChatCompletionOutputUsage(completion_tokens=10, prompt_tokens=31, total_tokens=41), object='chat.completion', prompt=[]))

In [ ]:
# This is where you can access the display:
print(os.environ.get('DLAI_LOCAL_URL').format(port='6006'))

## Setup ice cream production system

In [31]:
from smolagents import tool
from typing import Dict

menu_prices = {"crepe nutella": 1.50, "vanilla ice cream": 2, "maple pancake": 1.}

ORDER_BOOK = {}

@tool
def place_order(quantities: Dict[str, int], session_id: int) -> None:
    """Places a pre-order of snacks.

    Args:
        quantities: a dictionary with names as keys and quantities as values
        session_id: the id for the client session
    """
    global ORDER_BOOK
    assert isinstance(quantities, dict), "Incorrect type for the input dictionary!"
    assert [key in menu_prices for key in quantities.keys()], f"All food names should be within {menu_prices.keys()}"
    ORDER_BOOK[session_id] = quantities

@tool
def get_prices(quantities: Dict[str, int]) -> str:
    """Gets price for certain quantities of ice cream.

    Args:
        quantities: a dictionary with names as keys and quantities as values
    """
    assert isinstance(quantities, dict), "Incorrect type for the input dictionary!"
    assert [key in menu_prices for key in quantities.keys()], f"All food names should be within {menu_prices.keys()}"
    total_price = sum([menu_prices[key] * value for key, value in quantities.items()])
    return (
        f"Given the current menu prices:\n{menu_prices}\nThe total price for your order would be: ${total_price}"
    )

In [32]:
order_agent = CodeAgent(
    tools=[place_order, get_prices],
    model=HfApiModel("Qwen/Qwen2.5-Coder-32B-Instruct", provider="together")
)

In [33]:
order_agent.run(
    "Could I come and collect one crepe nutella?",
    additional_args={"session_id": 192}
)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Could I come and collect one crepe nutella?                                                                     │
│ You have been provided with these additional arguments, that you can access using the keys as variables in your │
│ python code:                                                                                                    │
│ {'session_id': 192}.                                                                                            │
│                                                                                                                 │
╰─ HfApiModel - Qwen/Qwen2.5-Coder-32B-Instruct ──────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Exception while exporting Span.
Traceback (most recent call last):
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
        (self._dns_host, self.port),
    ...<2 lines>...
        socket_options=self.socket_options,
    )
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
    ~~~~~~~~~~~~^^^^
ConnectionRefusedError: [Errno 61] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-p

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  place_order(quantities={'crepe nutella': 1}, session_id=session_id)                                              
  final_answer("Order placed for one crepe nutella.")                                                              
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Exception while exporting Span.
Traceback (most recent call last):
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
        (self._dns_host, self.port),
    ...<2 lines>...
        socket_options=self.socket_options,
    )
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
    ~~~~~~~~~~~~^^^^
ConnectionRefusedError: [Errno 61] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-p

Out - Final answer: Order placed for one crepe nutella.

Exception while exporting Span.
Traceback (most recent call last):
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
        (self._dns_host, self.port),
    ...<2 lines>...
        socket_options=self.socket_options,
    )
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
    ~~~~~~~~~~~~^^^^
ConnectionRefusedError: [Errno 61] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-p

[Step 1: Duration 1.47 seconds| Input tokens: 2,193 | Output tokens: 68]

Exception while exporting Span.
Traceback (most recent call last):
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
        (self._dns_host, self.port),
    ...<2 lines>...
        socket_options=self.socket_options,
    )
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
    ~~~~~~~~~~~~^^^^
ConnectionRefusedError: [Errno 61] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-p

'Order placed for one crepe nutella.'

### Try multiple orders

In [34]:
client_requests = [
    ("Could I come and collect one crepe nutella?", "place_order"),
    ("What would be the price for 1 crêpe nutella + 2 pancakes?", "get_prices"),
    ("How did you start your ice-cream business?", None),
    ("What's the weather at the Louvre right now?", None),
    ("I'm not sure if I should order. I want a vanilla ice cream. but if it's more expensive than $1, I don't want it. If it's below, I'll order it, please.", "place_order")
]

In [35]:
for request in client_requests:
    order_agent.run(
        request[0],
        additional_args={"session_id": 0, "menu_prices": menu_prices}
    )

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Could I come and collect one crepe nutella?                                                                     │
│ You have been provided with these additional arguments, that you can access using the keys as variables in your │
│ python code:                                                                                                    │
│ {'session_id': 0, 'menu_prices': {'crepe nutella': 1.5, 'vanilla ice cream': 2, 'maple pancake': 1.0}}.         │
│                                                                                                                 │
╰─ HfApiModel - Qwen/Qwen2.5-Coder-32B-Instruct ──────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Exception while exporting Span.
Traceback (most recent call last):
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
        (self._dns_host, self.port),
    ...<2 lines>...
        socket_options=self.socket_options,
    )
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
    ~~~~~~~~~~~~^^^^
ConnectionRefusedError: [Errno 61] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-p

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  quantities = {'crepe nutella': 1}                                                                                
  place_order(quantities=quantities, session_id=session_id)                                                        
  final_answer("Order placed for one crepe nutella.")                                                              
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Exception while exporting Span.
Traceback (most recent call last):
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
        (self._dns_host, self.port),
    ...<2 lines>...
        socket_options=self.socket_options,
    )
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/urllib3/util/connection.py", line 73, in create_connection
    sock.connect(sa)
    ~~~~~~~~~~~~^^^^
ConnectionRefusedError: [Errno 61] Connection refused

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-p

...

Error in generating model output:
402 Client Error: Payment Required for url: https://router.huggingface.co/together/v1/chat/completions (Request ID:
Root=1-681354bd-2ec5ad0c61c67a12291f7083;bd70c510-fda4-4da9-bd6a-0a805b9c3603)

You have exceeded your monthly included credits for Inference Providers. Subscribe to PRO to get 20x more monthly 
included credits.

Exception while exporting Span.
Traceback (most recent call last):
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/huggingface_hub/utils/_http.py", line 409, in hf_raise_for_status
    response.raise_for_status()
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/requests/models.py", line 1024, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 402 Client Error: Payment Required for url: https://router.huggingface.co/together/v1/chat/completions

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/smolagents/agents.py", line 1225, in step
    chat_message: ChatMessage = self.model(
                                ~~~~~~~~~~^
        self.input_me

[Step 1: Duration 0.30 seconds| Input tokens: 2,227 | Output tokens: 93]

Exception while exporting Span.
Traceback (most recent call last):
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/huggingface_hub/utils/_http.py", line 409, in hf_raise_for_status
    response.raise_for_status()
    ~~~~~~~~~~~~~~~~~~~~~~~~~^^
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/requests/models.py", line 1024, in raise_for_status
    raise HTTPError(http_error_msg, response=self)
requests.exceptions.HTTPError: 402 Client Error: Payment Required for url: https://router.huggingface.co/together/v1/chat/completions

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/rupesh.panwar/Documents/AI-Projects/Code-Agent-smoll/.venv/lib/python3.13/site-packages/smolagents/agents.py", line 1225, in step
    chat_message: ChatMessage = self.model(
                                ~~~~~~~~~~^
        self.input_me

AgentGenerationError: Error in generating model output:
402 Client Error: Payment Required for url: https://router.huggingface.co/together/v1/chat/completions (Request ID: Root=1-681354bd-2ec5ad0c61c67a12291f7083;bd70c510-fda4-4da9-bd6a-0a805b9c3603)

You have exceeded your monthly included credits for Inference Providers. Subscribe to PRO to get 20x more monthly included credits.

In [36]:
import phoenix as px

spans = px.Client().get_spans_dataframe(project_name=PROJECT_NAME)
spans.head(20)

Arize Phoenix is not running on http://127.0.0.1:6006. Launch Phoenix with `import phoenix as px; px.launch_app()`


ConnectError: [Errno 61] Connection refused

### Add processing to extract desired information

In [38]:
import pandas as pd
import json

agents = spans[spans['span_kind'] == 'AGENT'].copy()
agents['task'] = agents['attributes.input.value'].apply(
    lambda x: json.loads(x).get('task') if isinstance(x, str) else None
)

tools = spans.loc[
    spans['span_kind'] == 'TOOL',
    ["attributes.tool.name", "attributes.input.value", "context.trace_id"]
].copy()

tools_per_task = agents[
    ["name", "start_time", "task", "context.trace_id"]
].merge(
    tools,
    on="context.trace_id",
    how="left",
)
tools_per_task.head()

NameError: name 'spans' is not defined

### Now, compare tool calls with exected tool calls

In [ ]:
def score_request(expected_tool: str, tool_calls: list):
    if expected_tool is None:
        return tool_calls == set(["final_answer"])
    else:
        return expected_tool in tool_calls

results = []
for request, expected_tool in client_requests:
    tool_calls = set(tools_per_task.loc[tools_per_task["task"] == request, "attributes.tool.name"].tolist())
    results.append(
        {
            "request": request,
            "tool_calls_performed": tool_calls,
            "is_correct": score_request(expected_tool, tool_calls)
        }
    )
pd.DataFrame(results)